# Pilot · build the 500-answer bank (250 math + 250 code)

Runtime: **A100**. One fresh runtime. Needs two Colab secrets (key icon on the left):
- `HF_TOKEN`, with access to Llama and Gemma (their tokenizers are used for the length filter);
- `OPENROUTER_API_KEY`, for GPT-4.1-mini. The script stops at a **$5** total and retries 429s with backoff.

Steps: generate with Phi-4-mini, Granite-3.3-8B, OLMo-2-7B (vLLM) and GPT-4.1-mini (OpenRouter) → label math (boxed answer + Math-Verify) and code (EvalPlus base+plus tests, run twice) → matched pairs per (problem, generator) → drop answers too long for any verifier → draw 125 pairs per domain under the PLAN.md §2 caps. About an hour.

In [ ]:
BRANCH = "feature/pilot-bank"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q -r bank/requirements.txt evalplus pyarrow
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

## 1. Open generators (vLLM): math then code for each
Each call is its own process, so GPU memory is freed between models.

In [ ]:
for d in ("math", "code"):
    !python -m bank.generate --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/phi4mini_{d}/generations.jsonl --model microsoft/Phi-4-mini-instruct --tag phi4mini
    !python -m bank.generate --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/granite33_{d}/generations.jsonl --model ibm-granite/granite-3.3-8b-instruct --tag granite33
    !python -m bank.generate --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/olmo2_{d}/generations.jsonl --model allenai/OLMo-2-1124-7B-Instruct --tag olmo2

## 2. GPT-4.1-mini through OpenRouter
Resumable: if it stops on rate limits, rerun the cell and it continues where it left off. The spend so far is in `bank/pilot/api_spend.json`.

In [ ]:
for d in ("math", "code"):
    !python -m bank.generate_api --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/gpt41mini_{d}/generations.jsonl
!cat bank/pilot/api_spend.json | head -5

## 3. Merge each domain's answers

In [ ]:
import json, glob, os
for d in ("math", "code"):
    rows = []
    for path in sorted(glob.glob(f"bank/pilot/gen/*_{d}/generations.jsonl")):
        rows += [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
        print(path, sum(1 for _ in open(path, encoding="utf-8")))
    os.makedirs(f"bank/pilot/{d}", exist_ok=True)
    with open(f"bank/pilot/{d}/generations.jsonl", "w", encoding="utf-8", newline="\n") as f:
        for r in sorted(rows, key=lambda r: r["candidate_id"]):
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(d, len(rows), "answers")

## 4. Labels
Math: the boxed answer and Math-Verify must agree. Code: EvalPlus's base and plus tests, each program twice (about 20 minutes).

In [ ]:
!python -m bank.label --questions bank/pilot/questions_math.jsonl --generations bank/pilot/math/generations.jsonl --out bank/pilot/math/labeled.jsonl
!python -m bank.label_evalplus --questions bank/pilot/questions_code.jsonl --generations bank/pilot/code/generations.jsonl --out bank/pilot/code/labeled.jsonl

## 5. Matched pairs, length filter and the pilot draw

In [ ]:
for d in ("math", "code"):
    !python -m bank.make_pairs --split pilot --questions bank/pilot/questions_{d}.jsonl --generations bank/pilot/{d}/generations.jsonl --labeled bank/pilot/{d}/labeled.jsonl --out bank/pilot/{d}
!python -m bank.token_lengths bank/pilot/math/pilot_bank.jsonl bank/pilot/code/pilot_bank.jsonl --out bank/pilot/too_long.json
!python -m bank.sample_pilot --math bank/pilot/math --code bank/pilot/code --too-long bank/pilot/too_long.json --out bank/pilot

## 6. Generator accuracy (for the report)

In [ ]:
import collections
for d in ("math", "code"):
    lab = {r["candidate_id"]: r for r in map(json.loads, open(f"bank/pilot/{d}/labeled.jsonl", encoding="utf-8"))}
    gens = list(map(json.loads, open(f"bank/pilot/{d}/generations.jsonl", encoding="utf-8")))
    tally = collections.defaultdict(collections.Counter)
    for g in gens:
        l = lab[g["candidate_id"]]
        tally[g["generator"]]["usable" if l["status"] == "ok" else l["status"]] += 1
        tally[g["generator"]]["correct"] += l["label"] == 1
    for gen, c in tally.items():
        print(d, gen, dict(c), "accuracy among usable:", round(c["correct"] / max(1, c["usable"]), 3))

## Download the results
Zips `bank/pilot/`. Send the zip back, or unzip it into the repo at the same path.

In [ ]:
!cd /content/project && zip -qr /content/pilot_bank.zip bank/pilot
from google.colab import files
files.download("/content/pilot_bank.zip")